## 1. Setup e imports

In [ ]:
import sys
import pandas as pd
from dotenv import load_dotenv

# Raiz del proyecto (ejecutar desde notebooks/)
sys.path.append("../../")
load_dotenv("../../.env")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

from src.config.settings import COUNTRY
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.url_filters import filter_product_urls
from src.core.price_tracking import run_price_tracking
from src.utils.clean_model_name import clean_model_name
from src.utils.replace_model_name import map_model_name, load_mapping_file
from src.core.italika_pipeline import build_price_comparison

fr_scraper = ScrapingUtils()

## 2. Configuracion

In [ ]:
BRAND_NAME   = "Italika"
BRAND_URL    = "https://www.italika.mx/"
YEAR_SCRAPED = 2026
CSV_PATH = rf"C:\Users\JTRUJILLO\Desktop\utiles\Reportes\historical_data\src\data\prices\{COUNTRY}-prices.csv"

## 3. Cargar inventario

In [ ]:
df_inventory = pd.read_csv(CSV_PATH)
df_inventory = df_inventory[df_inventory["brand"] == BRAND_NAME]
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])]
df_inventory

## 4. Obtener URLs de producto

In [ ]:
# Descubrimiento dinamico de URLs desde el sitio
urls = fr_scraper.get_all_urls_from_website(BRAND_URL)
valid_urls = filter_product_urls(urls, exclude_terms=["morbidelli"])
print(f"URLs validas encontradas: {len(valid_urls)}")

In [ ]:
# Lista fija para pruebas puntuales (comentar/descomentar segun necesidad)
italika_urls = [
    "https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p",
    "https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p",
    "https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p",
    "https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p",
    "https://www.italika.mx/cuatrimoto-italika-atv-190-gris-34006332/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at135x-verde-34006711/p",
    "https://www.italika.mx/motocicleta-urbana-italika-280z-gris-34006326/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at125-grafito-34006712/p",
    "https://www.italika.mx/cuatrimoto-italika-atv200-naranja-34006718/p",
    "https://www.italika.mx/motocicleta-italika-150z-negra-con-azul-con-gps-34007056/p",
    "https://www.italika.mx/motocicleta-chopper-italika-tc-250-gris-34006691/p",
    "https://www.italika.mx/motoneta-italika-bit-150-blanca-con-negro-34006931/p",
    "https://www.italika.mx/motocicleta-doble-proposito-italika-dm200-con-gps-azul-34006255-1300703478/p",
    "https://www.italika.mx/cuatrimoto-italika-atv300-azul-34006717/p",
    "https://www.italika.mx/motoneta-italika-d125-azul-con-negro-34006289/p",
    "https://www.italika.mx/motoneta-italika-ws175-sport-grafito-34006561/p",
    "https://www.italika.mx/motoneta-italika-ws150-sport-marino-34006551/p",
    "https://www.italika.mx/motocicleta-italika-firebird-300-negra-con-gps-34007122/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-roja-34006714/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-negra-34006715/p",
    "https://www.italika.mx/motocicleta-doble-proposito-italika-dm150-con-gps-blanca-34006888/p",
    "https://www.italika.mx/motoneta-italika-d150-lt-negra-34005388/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at110-rt-azul-34006720/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-rt-250-con-gps-roja-34007114/p",
    "https://www.italika.mx/motocicleta-chopper-italika-tc-300-negra-34006362/p",
    "https://www.italika.mx/motocicleta-italika-200z-azul-con-gps-34006885/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-negra-34005157/p",
    "https://www.italika.mx/motocicleta-italika-250z-carbono-negra-con-gps-34007058/p",
    "https://www.italika.mx/motocicleta-cafe-racer-italika-sptfire-250-plata-34005271/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-gts-gris-con-gps-34007052/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-300-roja-con-negro-34004912/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-300r-azul-con-gris-34006484/p",
    "https://www.italika.mx/motoneta-italika-modena-175-con-gps-azul-34006716/p",
    "https://www.italika.mx/motocicleta-chopper-italika-rc250-negra-34005813/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at130-rt-blanca-34006713/p",
    "https://www.italika.mx/motoneta-italika-ds150-negra-34006307/p",
    "https://www.italika.mx/motocicleta-italika-ft150-gts-gris-con-amarillo-con-gps-34007053/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft250-gts-con-gps-azul-con-negro-34006392/p",
    "https://www.italika.mx/motocicleta-urbana-italika-280z-con-gps-gris-34006887/p",
    "https://www.italika.mx/motocicleta-urbana-italika-125z-gris-con-gps-34006883/p",
]

## 5. Ejecutar price tracking

In [ ]:
rows = run_price_tracking(
    urls=italika_urls,
    brand_name=BRAND_NAME,
)

## 6. Limpiar y mapear modelos

In [ ]:
df_scraped = pd.DataFrame(rows)
df_scraped["year_scraped"] = YEAR_SCRAPED

df_scraped["model_name_clean"] = df_scraped["model_name"].apply(clean_model_name)

mapeo = load_mapping_file(COUNTRY, BRAND_NAME)
df_scraped["model_mapped"] = df_scraped["model_name_clean"].apply(lambda x: map_model_name(x, mapeo))
df_scraped["model_mapped"] = df_scraped["model_mapped"].str.lower()

df_scraped

## 7. Merge con inventario (inspeccion)

In [ ]:
df_merged = pd.merge(
    df_scraped,
    df_inventory.assign(model_lower=df_inventory["model"].str.lower()),
    left_on="model_mapped",
    right_on="model_lower",
    how="left",
    indicator=True,
)
df_merged

## 8. Construir comparacion de precios

In [ ]:
df_final = build_price_comparison(df_scraped, df_inventory, COUNTRY)
df_final

## 9. Inspeccion

In [ ]:
# Modelos scrapeados sin match en inventario
sin_match = df_merged[df_merged["_merge"] == "left_only"][["model_name", "model_name_clean", "model_mapped"]].drop_duplicates()
print(f"Sin match en inventario: {len(sin_match)}")
sin_match

In [ ]:
# Filas sin codigo de inventario
df_final[df_final["code"].isna()]

## 10. Exportar

In [ ]:
output_file = f"{BRAND_NAME}_precios.csv"
df_final.to_csv(output_file, index=False)
print(f"Archivo exportado: {output_file} — {len(df_final)} filas")

---
## Anexo: correr sobre catalogo completo

In [ ]:
# # Descomentar para correr sobre todas las URLs validas del catalogo
# rows_full = run_price_tracking(
#     urls=valid_urls,
#     brand_name=BRAND_NAME,
# )
# df_full = pd.DataFrame(rows_full)
# print(f"Modelos procesados: {len(df_full)}")
# print(df_full["change_status"].value_counts().to_dict())
# df_full.head()